# Étape 1 — Téléchargement NASA POWER + Feature Engineering Externe

Source : NASA POWER API (publique, sans clé)  
Paramètres : T2M, T2M_MAX, T2M_MIN, PRECTOTCORR, RH2M, WS2M, ALLSKY_SFC_SW_DWN  
Stratégie : télécharger la série complète 2007-2023 par lieu (55 lieux × chunks annuels),  
mettre en cache sur disque, puis calculer les features par rolling window et anomalie climatologique.

In [ ]:
import json
import time
import hashlib
import requests
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import date, timedelta

DATA  = Path('../data')
CACHE = DATA / 'nasa_cache'
CACHE.mkdir(exist_ok=True)

SEED = 42

# NASA POWER parameters — 7 params (max 20 per request)
PARAMS = 'T2M,T2M_MAX,T2M_MIN,PRECTOTCORR,RH2M,WS2M,ALLSKY_SFC_SW_DWN'

train = pd.read_csv(DATA / 'Train.csv', parse_dates=['deathdate'])
test  = pd.read_csv(DATA / 'Test.csv',  parse_dates=['deathdate'])

locs = (
    pd.concat([train[['latitude','longitude']], test[['latitude','longitude']]])
    .drop_duplicates()
    .reset_index(drop=True)
    .round(6)
)
print(f'Lieux uniques : {len(locs)}')

# Date range needed : earliest deathdate - 90 days → latest deathdate
all_dates = pd.concat([train['deathdate'], test['deathdate']])
START_DATE = (all_dates.min() - pd.Timedelta(days=95)).date()
END_DATE   = all_dates.max().date()
print(f'Fenêtre de téléchargement : {START_DATE} → {END_DATE}')

In [ ]:
def cache_path(lat: float, lon: float, year: int) -> Path:
    key = f'{lat:.6f}_{lon:.6f}_{year}'
    return CACHE / f'{key}.json'


def fetch_nasa_power(lat: float, lon: float, start: str, end: str,
                     max_retries: int = 5) -> dict:
    """Single NASA POWER call with exponential backoff on 429/500."""
    url = (
        f'https://power.larc.nasa.gov/api/temporal/daily/point'
        f'?parameters={PARAMS}&community=AG'
        f'&longitude={lon}&latitude={lat}'
        f'&start={start}&end={end}&format=JSON'
    )
    for attempt in range(max_retries):
        try:
            r = requests.get(url, timeout=60)
            if r.status_code == 200:
                return r.json()
            elif r.status_code in (429, 500, 502, 503):
                wait = 2 ** attempt + 2
                print(f'  HTTP {r.status_code} — retry in {wait}s')
                time.sleep(wait)
            else:
                print(f'  Unexpected HTTP {r.status_code} for ({lat},{lon}) {start}-{end}')
                return {}
        except requests.RequestException as e:
            wait = 2 ** attempt + 2
            print(f'  Exception {e} — retry in {wait}s')
            time.sleep(wait)
    return {}


def download_location(lat: float, lon: float,
                       start_date: date, end_date: date,
                       sleep_between: float = 1.0) -> pd.DataFrame:
    """Download full time series for one location using yearly chunks (≤366d per call)."""
    frames = []
    year = start_date.year
    while year <= end_date.year:
        cp = cache_path(lat, lon, year)
        if cp.exists():
            with open(cp) as f:
                data = json.load(f)
        else:
            chunk_start = max(start_date, date(year, 1, 1))
            chunk_end   = min(end_date,   date(year, 12, 31))
            data = fetch_nasa_power(
                lat, lon,
                chunk_start.strftime('%Y%m%d'),
                chunk_end.strftime('%Y%m%d'),
            )
            with open(cp, 'w') as f:
                json.dump(data, f)
            time.sleep(sleep_between)

        if data and 'properties' in data:
            props = data['properties']['parameter']
            df_y  = pd.DataFrame(props)
            df_y.index = pd.to_datetime(df_y.index, format='%Y%m%d')
            frames.append(df_y)
        year += 1

    if not frames:
        return pd.DataFrame()
    ts = pd.concat(frames).sort_index()
    # NASA POWER uses -999 for missing
    ts = ts.replace(-999.0, np.nan)
    return ts

In [ ]:
# ── Main download loop ─────────────────────────────────────────────────────
print(f'Downloading {len(locs)} locations × ~{END_DATE.year - START_DATE.year + 1} years')
print(f'Expected API calls (cache misses only): up to {len(locs) * (END_DATE.year - START_DATE.year + 1)}')
print()

ts_store: dict[tuple, pd.DataFrame] = {}

for i, row in locs.iterrows():
    lat, lon = round(row['latitude'], 6), round(row['longitude'], 6)
    ts = download_location(lat, lon, START_DATE, END_DATE, sleep_between=1.2)
    ts_store[(lat, lon)] = ts
    print(f'  [{i+1:2d}/{len(locs)}] ({lat:.4f}, {lon:.4f}) → {len(ts)} jours')

print('\nTéléchargement terminé.')

In [ ]:
# ── Sample sanity check ───────────────────────────────────────────────────
sample_key = list(ts_store.keys())[0]
sample_ts  = ts_store[sample_key]
print(f'Location {sample_key} — {len(sample_ts)} jours')
print(sample_ts.head(3))
print()
print('RH2M stats:')
print(sample_ts['RH2M'].describe())

In [ ]:
# ── Feature engineering per (lat, lon, deathdate) ─────────────────────────
#
# For each record, we compute rolling windows on the 90-day window ending at deathdate:
#   - rh2m_7d, rh2m_30d, rh2m_90d     : mean relative humidity
#   - ws2m_7d, ws2m_30d                : mean wind speed
#   - solar_7d, solar_30d              : mean solar radiation
#   - heat_index_30d                   : mean Steadman heat index (T2M_MAX × RH2M)
#   - rh2m_anomaly_vs_clim             : rh2m_30d - monthly climatology mean
#   - temp_max_30d (NASA), temp_min_30d (NASA): cross-validation vs existing features
#   - is_high_humidity                 : rh2m_30d > 60 (Anopheles survival threshold)

def steadman_heat_index(T_c: pd.Series, RH: pd.Series) -> pd.Series:
    """Simplified Steadman heat index (°C input, RH in %). Valid when T>27°C."""
    T = T_c  # already Celsius from NASA POWER
    hi = (
        -8.78469475556
        + 1.61139411   * T
        + 2.33854883889 * RH
        - 0.14611605   * T  * RH
        - 0.012308094  * T  * T
        - 0.0164248278 * RH * RH
        + 0.002211732  * T  * T  * RH
        + 0.00072546   * T  * RH * RH
        - 0.000003582  * T  * T  * RH * RH
    )
    return hi


def compute_features_for_record(ts: pd.DataFrame, target_date: pd.Timestamp) -> dict:
    """Extract rolling features ending at target_date from a location time series."""
    # Window: [target_date - 90d, target_date]
    w90 = ts[ts.index <= target_date].tail(90)
    w30 = w90.tail(30)
    w14 = w90.tail(14)
    w7  = w90.tail(7)

    feat = {}

    # Relative humidity
    feat['rh2m_7d']  = w7['RH2M'].mean()  if len(w7)  > 0 else np.nan
    feat['rh2m_14d'] = w14['RH2M'].mean() if len(w14) > 0 else np.nan
    feat['rh2m_30d'] = w30['RH2M'].mean() if len(w30) > 0 else np.nan
    feat['rh2m_90d'] = w90['RH2M'].mean() if len(w90) > 0 else np.nan

    # Wind speed
    feat['ws2m_7d']  = w7['WS2M'].mean()  if len(w7)  > 0 else np.nan
    feat['ws2m_30d'] = w30['WS2M'].mean() if len(w30) > 0 else np.nan

    # Solar radiation (photosynthesis / heat stress proxy)
    feat['solar_7d']  = w7['ALLSKY_SFC_SW_DWN'].mean()  if len(w7)  > 0 else np.nan
    feat['solar_30d'] = w30['ALLSKY_SFC_SW_DWN'].mean() if len(w30) > 0 else np.nan

    # Heat index (30d mean)
    if len(w30) > 0 and w30['T2M_MAX'].notna().any() and w30['RH2M'].notna().any():
        hi_series = steadman_heat_index(w30['T2M_MAX'], w30['RH2M'])
        feat['heat_index_30d'] = hi_series.mean()
        feat['heat_index_max'] = hi_series.max()
    else:
        feat['heat_index_30d'] = np.nan
        feat['heat_index_max'] = np.nan

    # RH anomaly vs short term
    feat['rh2m_anomaly_7v30']  = feat['rh2m_7d']  - feat['rh2m_30d']
    feat['rh2m_anomaly_30v90'] = feat['rh2m_30d'] - feat['rh2m_90d']

    # High humidity flag (Anopheles needs >60% RH)
    feat['is_high_humidity_30d'] = int(feat['rh2m_30d'] > 60) if not np.isnan(feat['rh2m_30d']) else np.nan
    feat['is_high_humidity_7d']  = int(feat['rh2m_7d']  > 60) if not np.isnan(feat['rh2m_7d'])  else np.nan

    # NASA cross-check temperatures (for validation vs existing ERA5 features)
    feat['nasa_tmax_30d'] = w30['T2M_MAX'].mean() if len(w30) > 0 else np.nan
    feat['nasa_tmin_30d'] = w30['T2M_MIN'].mean() if len(w30) > 0 else np.nan
    feat['nasa_rain_30d'] = w30['PRECTOTCORR'].sum() if len(w30) > 0 else np.nan

    return feat

In [ ]:
# ── Compute climatological anomalies (monthly mean per location) ───────────
# Baseline: all available data (2007-2022) excluding the 30d window itself
# This gives the "expected" humidity for that month at that location.

clim_store: dict[tuple, pd.DataFrame] = {}

for (lat, lon), ts in ts_store.items():
    if ts.empty:
        continue
    monthly = ts[['RH2M', 'WS2M', 'T2M', 'T2M_MAX', 'PRECTOTCORR']].copy()
    monthly['month'] = monthly.index.month
    clim = monthly.groupby('month').mean()
    clim.columns = [f'clim_{c}' for c in clim.columns if c != 'month']
    clim_store[(lat, lon)] = clim

print(f'Climatologies calculées pour {len(clim_store)} lieux.')
# Sample
sample_key = list(clim_store.keys())[0]
print(clim_store[sample_key][['clim_RH2M', 'clim_T2M']].head(4))

In [ ]:
def get_clim_anomaly_features(lat: float, lon: float,
                               target_date: pd.Timestamp,
                               rh2m_30d: float) -> dict:
    """Compute anomaly vs monthly climatology for a given location and date."""
    feat = {}
    key = (round(lat, 6), round(lon, 6))
    if key not in clim_store:
        feat['rh2m_clim_month'] = np.nan
        feat['rh2m_clim_anomaly'] = np.nan
        return feat
    clim = clim_store[key]
    month = target_date.month
    if month in clim.index:
        feat['rh2m_clim_month'] = clim.loc[month, 'clim_RH2M']
        feat['rh2m_clim_anomaly'] = rh2m_30d - feat['rh2m_clim_month']
    else:
        feat['rh2m_clim_month']   = np.nan
        feat['rh2m_clim_anomaly'] = np.nan
    return feat

In [ ]:
# ── Build feature dataframe for all records ────────────────────────────────

def build_nasa_features(df: pd.DataFrame) -> pd.DataFrame:
    """Apply feature extraction for each row. Returns new features as a DataFrame."""
    rows = []
    for _, row in df.iterrows():
        lat  = round(row['latitude'],  6)
        lon  = round(row['longitude'], 6)
        ddate = row['deathdate']
        key  = (lat, lon)

        if key not in ts_store or ts_store[key].empty:
            rows.append({'ID': row['ID']})
            continue

        ts = ts_store[key]
        feat = compute_features_for_record(ts, ddate)

        # Climatological anomaly
        clim_feat = get_clim_anomaly_features(lat, lon, ddate, feat.get('rh2m_30d', np.nan))
        feat.update(clim_feat)

        feat['ID'] = row['ID']
        rows.append(feat)

    return pd.DataFrame(rows)


print('Calcul des features NASA POWER pour le train...')
nasa_train = build_nasa_features(train)
print(f'  Train : {nasa_train.shape}')

print('Calcul des features NASA POWER pour le test...')
nasa_test  = build_nasa_features(test)
print(f'  Test  : {nasa_test.shape}')

print()
print('Colonnes NASA features :')
print([c for c in nasa_train.columns if c != 'ID'])

In [ ]:
# ── NaN audit ─────────────────────────────────────────────────────────────
nasa_feat_cols = [c for c in nasa_train.columns if c != 'ID']
nan_rates = nasa_train[nasa_feat_cols].isnull().mean().sort_values(ascending=False)
print('Taux de NaN par feature NASA (train) :')
print(nan_rates[nan_rates > 0].to_string())
print()

# Fill NaN with column median for features with low NaN rate (<10%)
for col in nasa_feat_cols:
    if nasa_train[col].isnull().mean() < 0.10:
        med = nasa_train[col].median()
        nasa_train[col] = nasa_train[col].fillna(med)
        nasa_test[col]  = nasa_test[col].fillna(med)

print('NaN après imputation :', nasa_train[nasa_feat_cols].isnull().sum().sum())

In [ ]:
# ── Descriptive stats on new features ─────────────────────────────────────
print('Stats des nouvelles features (train) :')
print(nasa_train[nasa_feat_cols].describe().round(2).T[['mean','std','min','max']])

In [ ]:
# ── Correlation avec le target ────────────────────────────────────────────
y = train['is_climate_sensitive']
corrs = {}
for col in nasa_feat_cols:
    corrs[col] = nasa_train[col].corr(y)

corr_df = pd.Series(corrs).sort_values(key=abs, ascending=False)
print('Corrélation avec is_climate_sensitive :')
print(corr_df.to_string())

In [ ]:
# ── Open-Meteo cross-validation (Étape 2 — échantillon 10 lieux) ──────────
# Compare NASA POWER vs ERA5-based Open-Meteo on tmax_30d and rain_30d
# pour détecter les biais systématiques.

import requests

def fetch_open_meteo(lat: float, lon: float, start: str, end: str) -> pd.DataFrame:
    """Open-Meteo archive (ERA5). start/end format: YYYY-MM-DD."""
    url = (
        f'https://archive-api.open-meteo.com/v1/archive'
        f'?latitude={lat}&longitude={lon}'
        f'&start_date={start}&end_date={end}'
        f'&daily=temperature_2m_max,temperature_2m_min,precipitation_sum,wind_speed_10m_max'
        f'&timezone=UTC'
    )
    r = requests.get(url, timeout=60)
    if r.status_code != 200:
        return pd.DataFrame()
    data = r.json()
    df = pd.DataFrame(data['daily'])
    df['time'] = pd.to_datetime(df['time'])
    df = df.set_index('time')
    return df


# Sample 10 random locations
import random
random.seed(SEED)
sample_locs = locs.sample(10, random_state=SEED).reset_index(drop=True)

# Use a fixed 30-day window for comparison: 2015-01-01 to 2015-01-30
COMP_START = '2015-01-01'
COMP_END   = '2015-01-30'

comparison = []
for _, row in sample_locs.iterrows():
    lat, lon = round(row['latitude'], 6), round(row['longitude'], 6)

    # NASA POWER values
    key = (lat, lon)
    ts  = ts_store.get(key, pd.DataFrame())
    if not ts.empty:
        w = ts.loc['2015-01-01':'2015-01-30']
        nasa_tmax = w['T2M_MAX'].mean()
        nasa_rain = w['PRECTOTCORR'].sum()
    else:
        nasa_tmax = np.nan
        nasa_rain = np.nan

    # Open-Meteo values
    time.sleep(0.8)
    om = fetch_open_meteo(lat, lon, COMP_START, COMP_END)
    if not om.empty:
        om_tmax = om['temperature_2m_max'].mean()
        om_rain = om['precipitation_sum'].sum()
    else:
        om_tmax = np.nan
        om_rain = np.nan

    comparison.append({
        'lat': lat, 'lon': lon,
        'nasa_tmax_30d': nasa_tmax, 'om_tmax_30d': om_tmax,
        'delta_tmax': nasa_tmax - om_tmax,
        'nasa_rain_30d': nasa_rain, 'om_rain_30d': om_rain,
        'delta_rain': nasa_rain - om_rain,
    })

comp_df = pd.DataFrame(comparison)
print('Comparaison NASA POWER vs Open-Meteo (ERA5) — janv. 2015 :')
print(comp_df[['lat','lon','nasa_tmax_30d','om_tmax_30d','delta_tmax',
               'nasa_rain_30d','om_rain_30d','delta_rain']].round(2).to_string())
print()
print('Biais moyen Tmax (NASA - ERA5) :', comp_df['delta_tmax'].mean().round(3), '°C')
print('Biais moyen Rain (NASA - ERA5) :', comp_df['delta_rain'].mean().round(3), 'mm')

In [ ]:
# ── Merge avec les features existantes ────────────────────────────────────
X_train_old = pd.read_parquet(DATA / 'X_train.parquet')
X_test_old  = pd.read_parquet(DATA / 'X_test.parquet')
train_ids   = train['ID']
test_ids    = test['ID']

# Add ID columns back for merge
X_train_old['ID'] = train_ids.values
X_test_old['ID']  = test_ids.values

X_train_v2 = X_train_old.merge(nasa_train, on='ID', how='left').drop(columns=['ID'])
X_test_v2  = X_test_old.merge(nasa_test,   on='ID', how='left').drop(columns=['ID'])

print(f'X_train_v2 : {X_train_v2.shape}  (was {X_train_old.shape[1]-1} features)')
print(f'X_test_v2  : {X_test_v2.shape}')
print(f'NaN X_train_v2 : {X_train_v2.isnull().sum().sum()}')
print(f'NaN X_test_v2  : {X_test_v2.isnull().sum().sum()}')

In [ ]:
# ── Save enriched datasets ─────────────────────────────────────────────────
X_train_v2.to_parquet(DATA / 'X_train_v2.parquet')
X_test_v2.to_parquet(DATA / 'X_test_v2.parquet')

print('Sauvegardé :')
print('  data/X_train_v2.parquet')
print('  data/X_test_v2.parquet')
print()
print('Nouvelles features ajoutées :')
new_cols = [c for c in X_train_v2.columns if c not in X_train_old.columns]
for c in new_cols:
    print(f'  {c}')

In [ ]:
# ── Quick CV validation — does adding NASA features improve the score? ─────
# GroupKFold by location to stay consistent with modeling notebook.

import lightgbm as lgb
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, roc_auc_score

SEED = 42
y = pd.read_parquet(DATA / 'y_train.parquet')['is_climate_sensitive']
groups = train['location'].values

CAT_COLS = ['zone', 'gender']

def run_cv(X: pd.DataFrame, label: str, n_splits: int = 5, seed: int = SEED):
    Xc = X.copy()
    for col in CAT_COLS:
        if col in Xc.columns:
            le = LabelEncoder()
            Xc[col] = le.fit_transform(Xc[col].astype(str))

    gkf = GroupKFold(n_splits=n_splits)
    oof_proba = np.zeros(len(y))

    # Best params from Optuna (v4)
    params = dict(
        objective='binary', metric='binary_logloss', verbosity=-1,
        learning_rate=0.0351, num_leaves=117, min_child_samples=13,
        feature_fraction=0.771, bagging_fraction=0.995, bagging_freq=5,
        reg_alpha=0.00255, reg_lambda=0.0651,
        scale_pos_weight=(y == 0).sum() / (y == 1).sum(),
        n_estimators=2000, random_state=seed,
    )

    for fold, (tr_idx, va_idx) in enumerate(gkf.split(Xc, y, groups)):
        X_tr, X_va = Xc.iloc[tr_idx], Xc.iloc[va_idx]
        y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

        model = lgb.LGBMClassifier(**params)
        model.fit(
            X_tr, y_tr,
            eval_X=X_va, eval_y=y_va,
            callbacks=[
                lgb.early_stopping(200, verbose=False),
                lgb.log_evaluation(period=-1),
            ],
        )
        oof_proba[va_idx] = model.predict_proba(X_va)[:, 1]

    oof_pred = (oof_proba >= 0.5).astype(int)
    f1  = f1_score(y, oof_pred)
    auc = roc_auc_score(y, oof_proba)
    score = 0.6 * f1 + 0.4 * auc
    print(f'[{label}] F1={f1:.4f}  AUC={auc:.4f}  Score={score:.4f}')
    return score, oof_proba


print('CV baseline (features v1) :')
score_v1, oof_v1 = run_cv(X_train_old.drop(columns=['ID']), 'v1_baseline')
print()
print('CV avec NASA features (features v2) :')
score_v2, oof_v2 = run_cv(X_train_v2, 'v2_nasa')
print()
print(f'Delta Score : {score_v2 - score_v1:+.4f}')

In [ ]:
# ── Feature importance on new features ────────────────────────────────────
# Retrain on full data with best params, check new feature importances

Xc = X_train_v2.copy()
for col in CAT_COLS:
    if col in Xc.columns:
        le = LabelEncoder()
        Xc[col] = le.fit_transform(Xc[col].astype(str))

params_full = dict(
    objective='binary', metric='binary_logloss', verbosity=-1,
    learning_rate=0.0351, num_leaves=117, min_child_samples=13,
    feature_fraction=0.771, bagging_fraction=0.995, bagging_freq=5,
    reg_alpha=0.00255, reg_lambda=0.0651,
    scale_pos_weight=(y == 0).sum() / (y == 1).sum(),
    n_estimators=500, random_state=SEED,
)
model_full = lgb.LGBMClassifier(**params_full)
model_full.fit(Xc, y)

imp = pd.Series(model_full.feature_importances_, index=Xc.columns)
imp_new = imp[new_cols].sort_values(ascending=False)
print('Importance des nouvelles features NASA :')
print(imp_new.to_string())
print()
print('Top 20 features globales :')
print(imp.sort_values(ascending=False).head(20).to_string())

In [ ]:
# ── Log experiment in experiments.csv ─────────────────────────────────────
from datetime import date as dt_date

exp_path = Path('../notes/experiments.csv')
new_row = pd.DataFrame([{
    'date': str(dt_date.today()),
    'version': 'v6',
    'model': 'LGBM_Optuna_scale_pos_weight',
    'features': f'v2_nasa ({len(new_cols)} new features: RH2M, WS2M, HeatIndex, solar)',
    'cv_strategy': 'GroupKFold(5)_by_location',
    'oof_score': round(score_v2, 4),
    'lb_score': '',
    'delta_vs_prev': round(score_v2 - score_v1, 4),
    'notes': 'NASA POWER external data, 55 locations, 2007-2022, cached',
}])
if exp_path.exists():
    exp_df = pd.read_csv(exp_path)
    exp_df = pd.concat([exp_df, new_row], ignore_index=True)
else:
    exp_df = new_row
exp_df.to_csv(exp_path, index=False)
print('Expérience loguée dans notes/experiments.csv')
print(new_row.to_string(index=False))

In [ ]:
# ── Generate submission if CV improved ────────────────────────────────────
# Only generate if score_v2 > score_v1, as per external_data_prompt instructions.

if score_v2 > score_v1:
    print(f'Score amélioré (+{score_v2 - score_v1:.4f}) — génération de la soumission v6...')

    # Full retrain on v2 features
    Xc_train = X_train_v2.copy()
    Xc_test  = X_test_v2.copy()
    for col in CAT_COLS:
        le = LabelEncoder()
        Xc_train[col] = le.fit_transform(Xc_train[col].astype(str))
        Xc_test[col]  = le.transform(Xc_test[col].astype(str))

    params_submit = dict(
        objective='binary', metric='binary_logloss', verbosity=-1,
        learning_rate=0.0351, num_leaves=117, min_child_samples=13,
        feature_fraction=0.771, bagging_fraction=0.995, bagging_freq=5,
        reg_alpha=0.00255, reg_lambda=0.0651,
        scale_pos_weight=(y == 0).sum() / (y == 1).sum(),
        n_estimators=1000, random_state=SEED,
    )
    model_submit = lgb.LGBMClassifier(**params_submit)
    model_submit.fit(Xc_train, y)

    proba_test = model_submit.predict_proba(Xc_test)[:, 1]
    pred_test  = (proba_test >= 0.5).astype(int)

    test_ids_ser = pd.read_parquet(DATA / 'test_ids.parquet')['ID']
    sub = pd.DataFrame({
        'ID': test_ids_ser,
        'TargetF1': pred_test,
        'TargetRAUC': proba_test,
    })

    # Validate format
    sample = pd.read_csv(DATA / 'SampleSubmission.csv')
    assert list(sub.columns) == list(sample.columns), 'Format mismatch!'
    assert len(sub) == len(sample), 'Length mismatch!'

    out_path = DATA / f'submission_v6_nasa_{score_v2:.4f}.csv'
    sub.to_csv(out_path, index=False)
    print(f'Soumission sauvegardée : {out_path}')
    print(f'TargetF1 distribution : {sub["TargetF1"].value_counts().to_dict()}')
else:
    print(f'Score non amélioré ({score_v2:.4f} vs {score_v1:.4f}) — pas de soumission.')
    print('Les features NASA POWER ne sont pas informatives sur ce CV. Vérifier les NaN et la qualité des données.')